# Mini-projet 01 — Simulateur d'impôt sur le revenu

Application des conditions et de la validation : calcul de l'impôt sur le revenu français selon le barème 2024, avec quotient familial et tranches progressives.

Les tests utilisent `pytest.approx` pour les comparaisons de flottants — exécute la cellule d'import au début.

### Setup
Exécute la cellule suivante avant tout :

In [1]:
import pytest

---
## 📖 Contexte

_Lis cette partie avant d'attaquer les exercices._

Maintenant que tu maîtrises les **variables**, les **conditions** et la **validation d'entrée**,
on va t'en faire un usage concret : un simulateur d'**impôt sur le revenu** à la française.

Le calcul de l'impôt repose presque entièrement sur des **conditions** (tranches d'imposition,
situation familiale, abattements). C'est l'occasion idéale de t'entraîner sur des `if / elif / else`
sur un cas réel.

> ⚠️ **Avertissement** : ce simulateur est volontairement **simplifié à des fins pédagogiques**.
> Il ne remplace pas le simulateur officiel des impôts. On ignore plein de cas réels (CSG, parts
> majorées, plafonnement du quotient familial, crédits d'impôt, etc.).

### Concepts ré-utilisés

- expressions arithmétiques (`+`, `-`, `*`, `/`)
- `if / elif / else`
- validation d'entrée (`raise ValueError`)
- comparaisons (`<`, `>`, `==`)
- fonctions retournant un nombre

### Contexte du calcul

Le calcul officiel se fait en 4 étapes :

1. **Revenu net imposable** = salaire brut − frais professionnels.
   - Par défaut, l'État applique un abattement forfaitaire de **10 %** des salaires.
   - Si tes frais réels sont **supérieurs** à cet abattement, tu peux les déclarer à la place.
2. **Nombre de parts fiscales** : dépend de la situation familiale (célibataire, marié·e, veuf·ve)
   et du nombre d'enfants à charge.
3. **Quotient familial** = revenu net imposable / nombre de parts.
4. **Impôt** = barème progressif appliqué au quotient familial, puis multiplié par le nombre de parts.

### Barème 2024 (par part)

| Tranche                           | Taux  |
|-----------------------------------|-------|
| Jusqu'à 11 294 €                  | 0 %   |
| De 11 295 € à 28 797 €            | 11 %  |
| De 28 798 € à 82 341 €            | 30 %  |
| De 82 342 € à 177 106 €           | 41 %  |
| Au-delà de 177 106 €              | 45 %  |

L'impôt est calculé **par tranche** : seule la partie du revenu **dans** une tranche est taxée
au taux de cette tranche. Exemple pour un quotient familial de 30 000 € :

```
Tranche 1 (0 → 11 294)      : 0 € d'impôt
Tranche 2 (11 295 → 28 797) : (28 797 − 11 294) × 11 % = 1 925,33 €
Tranche 3 (28 798 → 30 000) : (30 000 − 28 797) × 30 % =   360,90 €
                                                     ─────────────
Total                                                = 2 286,23 €
```

### Nombre de parts (version simplifiée)

| Situation        | Parts de base |
|------------------|---------------|
| `"celibataire"`  | 1             |
| `"marie"`        | 2             |
| `"veuf"`         | 1             |

Auquel on ajoute pour les enfants :
- 0,5 part pour chacun des **2 premiers** enfants
- 1 part pour chaque enfant **à partir du 3ᵉ**

Exemple : un couple avec 3 enfants → 2 + 0,5 + 0,5 + 1 = **4 parts**.

---
## 🎯 Fonctions à implémenter

Pour chaque fonction, lis l'énoncé, complète le stub, puis exécute la cellule de tests.

### `calculer_revenu_net_imposable`

- Si `salaire_brut < 0`, lever une `ValueError`.
- Si `frais_reels < 0`, lever une `ValueError`.
- Calcule l'**abattement de 10 %** sur le salaire brut.
- Retourne le salaire brut **moins le plus élevé** entre l'abattement forfaitaire et les frais réels.

In [3]:
def calculer_revenu_net_imposable(salaire_brut: float, frais_reels: float = 0.0) -> float:
    """Retourne le revenu net imposable.

    Règle :
        - Abattement forfaitaire de 10 % sur le salaire brut.
        - Si les frais réels sont supérieurs à l'abattement, on les utilise à la place.
        - Le revenu net imposable = salaire brut - max(abattement, frais_reels).

    Erreurs :
        - ValueError si salaire_brut ou frais_reels est négatif.
    """
    # TODO
    if (salaire_brut < 0 or frais_reels < 0):
        raise ValueError
    else:
        return salaire_brut - max((salaire_brut/10), frais_reels)
    raise NotImplementedError

### `calculer_nombre_de_parts`

- Si `nb_enfants < 0`, lever une `ValueError`.
- Si `situation` n'est pas `"celibataire"`, `"marie"` ou `"veuf"`, lever une `ValueError`.
- Retourne le nombre de parts selon le tableau ci-dessus.

In [4]:
def calculer_nombre_de_parts(situation: str, nb_enfants: int) -> float:
    """Retourne le nombre de parts fiscales du foyer.

    Parts de base :
        - "celibataire" -> 1
        - "marie"       -> 2
        - "veuf"        -> 1

    Parts pour enfants :
        - 0,5 part pour chacun des 2 premiers enfants
        - 1 part pour chaque enfant à partir du 3e

    Erreurs :
        - ValueError si nb_enfants est négatif.
        - ValueError si situation n'est pas une des trois valeurs attendues.
    """
    # TODO
    parts = 0
    situations = {"celibataire", "marie", "veuf"}
    if (nb_enfants < 0 or situation not in situations):
        raise ValueError
    else:
        if (situation == "marie"):
            parts = parts + 2
        else:
            parts = parts + 1
        for i in range(1, nb_enfants+1):
            if ( i == 1 or i == 2):
                parts = parts + 0.5
            else:
                parts = parts + 1
    return parts
    raise NotImplementedError

### `calculer_impot_par_tranches`

- Si `quotient_familial < 0`, lever une `ValueError`.
- Applique **chaque tranche** du barème en utilisant des `if` successifs.
- Retourne l'impôt **pour une part**.

> 💡 Astuce : pour la tranche `[11 295 → 28 797]`, tu calcules `(min(qf, 28797) − 11 294) × 0,11`,
> mais **uniquement si** `qf > 11 294`. Et ainsi de suite pour les tranches suivantes.

In [40]:
def calculer_impot_par_tranches(quotient_familial: float) -> float:
    """Retourne l'impôt pour UNE part en appliquant le barème 2024 :

        Jusqu'à 11 294 €               : 0 %
        De 11 295 € à 28 797 €         : 11 %
        De 28 798 € à 82 341 €         : 30 %
        De 82 342 € à 177 106 €        : 41 %
        Au-delà de 177 106 €           : 45 %

    Astuce : pour chaque tranche, on calcule la portion du quotient familial qui tombe
    dans cette tranche, puis on la multiplie par le taux. On additionne le tout.

    Erreurs :
        - ValueError si quotient_familial est négatif.
    """
    # TODO
    impo = 0.0
    if (quotient_familial < 0):
        raise ValueError
    else: 
        if (quotient_familial <= 11294):
            return 0
            
        if (quotient_familial <= 28797):
            impo = (min(quotient_familial, 28797) - 11294) * 0.11
            return impo
            
        if (quotient_familial <= 82341):
            return  (min(quotient_familial,82341) -  28797 )* 0.3 + (min(quotient_familial, 28797) - 11294) * 0.11
            
        if (quotient_familial <= 177106):
            return (min(quotient_familial,177106) - 82341)* 0.41  + (min(quotient_familial,82341) - 28798)* 0.3 + (min(quotient_familial, 28797) - 11294) * 0.11
            
        if (quotient_familial > 177106):
            impo = ((quotient_familial - 177106)* 0.45 + (min(quotient_familial,177106) - 82341) * 0.41  + (min(quotient_familial,82341) - 28798)* 0.3 + (min(quotient_familial, 28797) - 11294) * 0.11)
            return impo
    raise NotImplementedError

### `simuler_impot`

- Combine les 3 fonctions précédentes pour calculer l'impôt **total** du foyer.
- Retourne l'impôt total (= impôt par part × nombre de parts).

In [7]:
def simuler_impot(
    salaire_brut: float,
    situation: str,
    nb_enfants: int,
    frais_reels: float = 0.0,
) -> float:
    """Calcule l'impôt total du foyer en combinant les 3 fonctions précédentes.

    Étapes :
        1. Calculer le revenu net imposable.
        2. Calculer le nombre de parts.
        3. Calculer le quotient familial = revenu net imposable / nombre de parts.
        4. Calculer l'impôt pour une part via le barème.
        5. Multiplier par le nombre de parts -> impôt total du foyer.
    """
    # TODO
    valeur = calculer_revenu_net_imposable(salaire_brut,frais_reels)
    nbparts = calculer_nombre_de_parts(situation, nb_enfants)
    qf = valeur / nbparts
    part = calculer_impot_par_tranches(qf)
    return part * nbparts
    raise NotImplementedError

---
## Tests

### Revenu Net Imposable

In [8]:
def _test_abattement_forfaitaire_10_pourcent():
    assert calculer_revenu_net_imposable(30_000) == pytest.approx(27_000)
_test_abattement_forfaitaire_10_pourcent()

def _test_frais_reels_inferieurs_garde_abattement():
    assert calculer_revenu_net_imposable(30_000, frais_reels=2_000) == pytest.approx(27_000)
_test_frais_reels_inferieurs_garde_abattement()

def _test_frais_reels_superieurs_remplacent_abattement():
    assert calculer_revenu_net_imposable(30_000, frais_reels=5_000) == pytest.approx(25_000)
_test_frais_reels_superieurs_remplacent_abattement()

def _test_salaire_zero():
    assert calculer_revenu_net_imposable(0) == pytest.approx(0)
_test_salaire_zero()

def _test_salaire_negatif_leve_erreur():
    try:
        calculer_revenu_net_imposable(-1_000)
        raise AssertionError('attendait ValueError')
    except ValueError:
        pass
_test_salaire_negatif_leve_erreur()

def _test_frais_reels_negatifs_levent_erreur():
    try:
        calculer_revenu_net_imposable(30_000, frais_reels=-100)
        raise AssertionError('attendait ValueError')
    except ValueError:
        pass
_test_frais_reels_negatifs_levent_erreur()
print("✅ Revenu Net Imposable : OK")

✅ Revenu Net Imposable : OK


### Nombre De Parts

In [9]:
def _test_celibataire_sans_enfant():
    assert calculer_nombre_de_parts("celibataire", 0) == 1.0
_test_celibataire_sans_enfant()

def _test_marie_sans_enfant():
    assert calculer_nombre_de_parts("marie", 0) == 2.0
_test_marie_sans_enfant()

def _test_veuf_sans_enfant():
    assert calculer_nombre_de_parts("veuf", 0) == 1.0
_test_veuf_sans_enfant()

def _test_celibataire_un_enfant():
    assert calculer_nombre_de_parts("celibataire", 1) == 1.5
_test_celibataire_un_enfant()

def _test_marie_deux_enfants():
    assert calculer_nombre_de_parts("marie", 2) == 3.0
_test_marie_deux_enfants()

def _test_marie_trois_enfants():
    assert calculer_nombre_de_parts("marie", 3) == 4.0
_test_marie_trois_enfants()

def _test_marie_cinq_enfants():
    assert calculer_nombre_de_parts("marie", 5) == 6.0
_test_marie_cinq_enfants()

def _test_situation_inconnue_leve_erreur():
    try:
        calculer_nombre_de_parts("pacse", 0)
        raise AssertionError('attendait ValueError')
    except ValueError:
        pass
_test_situation_inconnue_leve_erreur()

def _test_nb_enfants_negatif_leve_erreur():
    try:
        calculer_nombre_de_parts("celibataire", -1)
        raise AssertionError('attendait ValueError')
    except ValueError:
        pass
_test_nb_enfants_negatif_leve_erreur()
print("✅ Nombre De Parts : OK")

✅ Nombre De Parts : OK


### Impot Par Tranches

In [41]:
def _test_revenu_dans_premiere_tranche_zero_impot():
    assert calculer_impot_par_tranches(10_000) == pytest.approx(0)
_test_revenu_dans_premiere_tranche_zero_impot()

def _test_pile_a_la_borne_de_la_premiere_tranche():
    assert calculer_impot_par_tranches(11_294) == pytest.approx(0)
_test_pile_a_la_borne_de_la_premiere_tranche()

def _test_revenu_dans_deuxieme_tranche():
    assert calculer_impot_par_tranches(20_000) == pytest.approx(957.66, rel=1e-3)
_test_revenu_dans_deuxieme_tranche()

def _test_revenu_dans_troisieme_tranche():
    assert calculer_impot_par_tranches(50_000) == pytest.approx(8_286.23, rel=1e-3)
_test_revenu_dans_troisieme_tranche()

def _test_revenu_dans_quatrieme_tranche():
    assert calculer_impot_par_tranches(100_000) == pytest.approx(25_228.72, rel=1e-3)
_test_revenu_dans_quatrieme_tranche()

def _test_revenu_dans_cinquieme_tranche():
    assert calculer_impot_par_tranches(200_000) == pytest.approx(67_144.48, rel=1e-3)
_test_revenu_dans_cinquieme_tranche()

def _test_quotient_familial_negatif_leve_erreur():
    try:
        calculer_impot_par_tranches(-1)
        raise AssertionError('attendait ValueError')
    except ValueError:
        pass
_test_quotient_familial_negatif_leve_erreur()
print("✅ Impot Par Tranches : OK")

✅ Impot Par Tranches : OK


### Simuler Impot

In [42]:
def _test_celibataire_smic_environ():
    assert simuler_impot(22_000, "celibataire", 0) == pytest.approx(935.66, rel=1e-3)
_test_celibataire_smic_environ()

def _test_couple_marie_avec_deux_enfants():
    assert simuler_impot(60_000, "marie", 2) == pytest.approx(2_212.98, rel=1e-3)
_test_couple_marie_avec_deux_enfants()

def _test_celibataire_haut_revenu():
    assert simuler_impot(100_000, "celibataire", 0) == pytest.approx(21_128.72, rel=1e-3)
_test_celibataire_haut_revenu()

def _test_avec_frais_reels():
    assert simuler_impot(40_000, "celibataire", 1, frais_reels=5_000) == pytest.approx(1_986.50, rel=1e-3)
_test_avec_frais_reels()
print("✅ Simuler Impot : OK")

✅ Simuler Impot : OK


---
## 🎁 Bonus pour aller plus loin

Si tu veux pousser le projet :

1. **Décote** : si l'impôt brut est inférieur à 1 929 € (célibataire) ou 3 191 € (marié·e),
   on applique une décote `min(impot, plafond) - 0,4525 × impot`. Implémente
   `appliquer_decote(impot, situation)`.
2. **Comparateur** : écris une fonction `comparer_profils` qui prend deux profils et indique
   lequel paie le plus d'impôt (et de combien).
3. **CLI** : crée un petit script `main.py` qui demande à l'utilisateur ses informations via
   `input()` et affiche le résultat formaté.